In [8]:
import pandas as pd
import numpy as np

def clean_healthcare_data(file_path):
    """
    Ingests and cleans raw healthcare data from a CSV file.
    """
    # 1. Ingest Data
    df = pd.read_csv(file_path)
    print(f"Initial row count: {len(df)}")
    
    # 2. Deduplication
    df = df.drop_duplicates()
    print(f"Row count after removing duplicates: {len(df)}")
    
    # 3. Anonymization (Drop PII)
    if 'patient_name' in df.columns:
        df = df.drop(columns=['patient_name'])
        
    # 4. Handle Age Outliers and Missing Values
    # Convert to numeric, pushing non-numbers to NaN
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    
    # Flag impossible ages (over 110 or negative) as NaN
    df.loc[(df['age'] > 110) | (df['age'] < 0), 'age'] = np.nan
    
    # Impute missing ages with the median
    df['age'] = df['age'].fillna(df['age'].median()).round(1)
    
    # 5. Standardize Clinical Strings (Blood Pressure)
    # Use Regex to enforce the "number/number" format; otherwise make it NaN
    valid_bp_pattern = r'^\d{2,3}/\d{2,3}$'
    df.loc[~df['blood_pressure'].astype(str).str.match(valid_bp_pattern, na=False), 'blood_pressure'] = np.nan
    
    # 6. Standardize Date Formats
    # format='mixed' parses both YYYY-MM-DD and DD-MM-YYYY formats, standardizing to ISO 8601
    df['visit_date'] = pd.to_datetime(df['visit_date'], format='mixed').dt.strftime('%Y-%m-%d')
    
    return df

# --- Execute the Pipeline ---
if __name__ == "__main__":
    # File path (string)
    input_file = '/Users/kalyanram/Downloads/Healthcare Data Cleaning Pipeline.csv'
    
    # Load the raw data just to print the "Before" stats
    raw_df = pd.read_csv(input_file)
    print("=== BEFORE CLEANING ===")
    print(f"Total Rows: {len(raw_df)}")
    print("\nMissing Values Count:")
    print(raw_df.isna().sum())  

    # Run the cleaning function (which expects the file path string)
    cleaned_df = clean_healthcare_data(input_file)

    print("\n\n=== AFTER CLEANING ===")
    print(f"Total Rows: {len(cleaned_df)}")
    print("\nMissing Values Count (Post-Imputation & Standardization):")
    print(cleaned_df.isna().sum())
    print("\nSample of Cleaned Data:")
    print(cleaned_df.head())

=== BEFORE CLEANING ===
Total Rows: 1050

Missing Values Count:
patient_id         0
patient_name       0
age               33
blood_pressure    11
visit_date         0
dtype: int64
Initial row count: 1050
Row count after removing duplicates: 1000


=== AFTER CLEANING ===
Total Rows: 1000

Missing Values Count (Post-Imputation & Standardization):
patient_id         0
age                0
blood_pressure    40
visit_date         0
dtype: int64

Sample of Cleaned Data:
  patient_id   age blood_pressure  visit_date
0     P-0001  57.5          97/85  2023-04-05
1     P-0002  47.9         143/82  2023-01-24
2     P-0003  59.7         127/85  2023-08-03
3     P-0004  72.8         137/65  2023-05-06
4     P-0005  46.5         116/65  2023-10-14


In [10]:
import pandas as pd
import numpy as np

def clean_healthcare_data(file_path):
    """
    Ingests, cleans, and imputes raw healthcare data from a CSV file.
    """
    df = pd.read_csv(file_path)
    
    # 1. Deduplication
    df = df.drop_duplicates()
    
    # 2. Anonymization
    if 'patient_name' in df.columns:
        df = df.drop(columns=['patient_name'])
        
    # 3. Handle Age
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df.loc[(df['age'] > 110) | (df['age'] < 0), 'age'] = np.nan
    df['age'] = df['age'].fillna(df['age'].median()).round(1)
    
    # 4. Standardize and Impute Blood Pressure
    # Blank out invalid strings first
    valid_bp_pattern = r'^\d{2,3}/\d{2,3}$'
    df.loc[~df['blood_pressure'].astype(str).str.match(valid_bp_pattern, na=False), 'blood_pressure'] = np.nan
    
    # Split the valid 'sys/dia' strings into two temporary numeric columns
    bp_split = df['blood_pressure'].str.split('/', expand=True)
    df['sys'] = pd.to_numeric(bp_split[0], errors='coerce')
    df['dia'] = pd.to_numeric(bp_split[1], errors='coerce')
    
    # Fill missing systolic and diastolic values with their respective medians
    df['sys'] = df['sys'].fillna(df['sys'].median())
    df['dia'] = df['dia'].fillna(df['dia'].median())
    
    # Recombine them into the original 'blood_pressure' format and drop temp columns
    df['blood_pressure'] = df['sys'].astype(int).astype(str) + '/' + df['dia'].astype(int).astype(str)
    df = df.drop(columns=['sys', 'dia'])
    
    # 5. Standardize Date Formats
    df['visit_date'] = pd.to_datetime(df['visit_date'], format='mixed').dt.strftime('%Y-%m-%d')
    
    return df

# --- Execute the Pipeline ---
if __name__ == "__main__":
    # Update with your file path
    input_file = '/Users/kalyanram/Downloads/Healthcare Data Cleaning Pipeline.csv'
    
    raw_df = pd.read_csv(input_file)
    print("=== BEFORE CLEANING ===")
    print(f"Total Rows: {len(raw_df)}")
    print("\nMissing Values Count:")
    print(raw_df.isna().sum())  

    # Run the cleaning function 
    cleaned_df = clean_healthcare_data(input_file)

    print("\n\n=== AFTER CLEANING ===")
    print(f"Total Rows: {len(cleaned_df)}")
    print("\nMissing Values Count (Post-Imputation):")
    print(cleaned_df.isna().sum())
    print("\nSample of Cleaned Data:")
    print(cleaned_df.head())

=== BEFORE CLEANING ===
Total Rows: 1050

Missing Values Count:
patient_id         0
patient_name       0
age               33
blood_pressure    11
visit_date         0
dtype: int64


=== AFTER CLEANING ===
Total Rows: 1000

Missing Values Count (Post-Imputation):
patient_id        0
age               0
blood_pressure    0
visit_date        0
dtype: int64

Sample of Cleaned Data:
  patient_id   age blood_pressure  visit_date
0     P-0001  57.5          97/85  2023-04-05
1     P-0002  47.9         143/82  2023-01-24
2     P-0003  59.7         127/85  2023-08-03
3     P-0004  72.8         137/65  2023-05-06
4     P-0005  46.5         116/65  2023-10-14


In [12]:
import pandas as pd
import numpy as np

def clean_healthcare_data(file_path):
    df = pd.read_csv(file_path)
    
    df = df.drop_duplicates()
    
    if 'patient_name' in df.columns:
        df = df.drop(columns=['patient_name'])
        
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df.loc[(df['age'] > 110) | (df['age'] < 0), 'age'] = np.nan
    df['age'] = df['age'].fillna(df['age'].median()).round(1)
    
    valid_bp_pattern = r'^\d{2,3}/\d{2,3}$'
    df.loc[~df['blood_pressure'].astype(str).str.match(valid_bp_pattern, na=False), 'blood_pressure'] = np.nan
    
    bp_split = df['blood_pressure'].str.split('/', expand=True)
    df['sys'] = pd.to_numeric(bp_split[0], errors='coerce')
    df['dia'] = pd.to_numeric(bp_split[1], errors='coerce')
    
    df['sys'] = df['sys'].fillna(df['sys'].median())
    df['dia'] = df['dia'].fillna(df['dia'].median())
    
    df['blood_pressure'] = df['sys'].astype(int).astype(str) + '/' + df['dia'].astype(int).astype(str)
    df = df.drop(columns=['sys', 'dia'])
    
    df['visit_date'] = pd.to_datetime(df['visit_date'], format='mixed').dt.strftime('%Y-%m-%d')
    
    return df

if __name__ == "__main__":
    input_file = '/Users/kalyanram/Downloads/Healthcare Data Cleaning Pipeline.csv'
    
    raw_df = pd.read_csv(input_file)
    print("=== BEFORE CLEANING ===")
    print(f"Total Rows: {len(raw_df)}")
    print("\nMissing Values Count:")
    print(raw_df.isna().sum())  

    cleaned_df = clean_healthcare_data(input_file)

    print("\n\n=== AFTER CLEANING ===")
    print(f"Total Rows: {len(cleaned_df)}")
    print("\nMissing Values Count (Post-Imputation):")
    print(cleaned_df.isna().sum())
    print("\nSample of Cleaned Data:")
    print(cleaned_df.head())

=== BEFORE CLEANING ===
Total Rows: 1050

Missing Values Count:
patient_id         0
patient_name       0
age               33
blood_pressure    11
visit_date         0
dtype: int64


=== AFTER CLEANING ===
Total Rows: 1000

Missing Values Count (Post-Imputation):
patient_id        0
age               0
blood_pressure    0
visit_date        0
dtype: int64

Sample of Cleaned Data:
  patient_id   age blood_pressure  visit_date
0     P-0001  57.5          97/85  2023-04-05
1     P-0002  47.9         143/82  2023-01-24
2     P-0003  59.7         127/85  2023-08-03
3     P-0004  72.8         137/65  2023-05-06
4     P-0005  46.5         116/65  2023-10-14
